### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors think recommendations are about predicting ratings. Seniors know they're about predicting ENGAGEMENT — what will a user actually click, watch, or buy? Predicted ratings ≠ predicted behavior. A user might rate arthouse films 5 stars but actually watch action movies every night. Production RecSys optimizes for engagement, not ratings.

**Mental Model:** Think of a librarian who knows 1,000 regular visitors. Content-based is like saying "you liked sci-fi books, here are more sci-fi books." Collaborative filtering is like saying "readers similar to you also loved THIS book" — even if it's not sci-fi. The librarian uses BOTH approaches, and so does every production RecSys.

**Common Junior Pitfall:** Evaluating a RecSys by accuracy alone ("did we predict the correct rating?"). In production, you evaluate by business metrics: click-through rate, watch time, purchase conversion. A model can be highly accurate on ratings but fail to generate clicks.

---

## 📑 Table of Contents

  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · The Recommendation Problem](#1-the-recommendation-problem)
  * [The User-Item Matrix](#the-user-item-matrix)
  * [Three Paradigms](#three-paradigms)
* [2 · Content-Based Filtering](#2-content-based-filtering)
  * [Idea](#idea)
  * [How It Works](#how-it-works)
* [3 · Collaborative Filtering](#3-collaborative-filtering)
  * [Idea](#idea)
  * [Two Flavors](#two-flavors)
* [4 · Evaluation Metrics](#4-evaluation-metrics)
  * [Rating Prediction Metrics](#rating-prediction-metrics)
  * [Ranking Metrics (More Important in Production!)](#ranking-metrics-more-important-in-production)
* [🎓 DEEP QUESTION ANSWERED](#deep-question-answered)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: Content-based vs collaborative filtering](#q1-content-based-vs-collaborative-filtering)
  * [Q2: User-based vs item-based CF](#q2-user-based-vs-item-based-cf)
  * [Q3: Evaluation trap](#q3-evaluation-trap)
* [🔨 Practical Practice](#practical-practice)
  * [Exercise 1: Item-Based CF](#exercise-1-item-based-cf)
  * [Exercise 2: Hybrid Recommender](#exercise-2-hybrid-recommender)
  * [Exercise 3: Cold Start Solution](#exercise-3-cold-start-solution)


In [ ]:
!pip install -q numpy pandas matplotlib scikit-learn

## 1 · The Recommendation Problem

### The User-Item Matrix

```
              Item 1  Item 2  Item 3  Item 4  Item 5
    User A  [  5       ?       3       ?       ? ]
    User B  [  ?       4       ?       2       ? ]
    User C  [  4       ?       5       ?       1 ]
    User D  [  ?       3       ?       ?       4 ]
```

The matrix is **extremely sparse** — typically 99%+ of entries are missing. The goal: fill in the `?` values to predict what each user would rate each item.

### Three Paradigms

| Approach | Input | Idea | Limitation |
|----------|-------|------|------------|
| **Content-Based** | Item features (genre, director) | "You liked action → more action" | No serendipity |
| **Collaborative** | User-item interactions | "Similar users liked this" | Cold start |
| **Hybrid** | Both | Best of both worlds | More complex |

In [ ]:
import numpy as np
import pandas as pd

# Create a realistic movie rating dataset
np.random.seed(42)

n_users = 200
n_items = 50

# Movie metadata
genres = ['Action', 'Comedy', 'Drama', 'Sci-Fi', 'Horror', 'Romance']
movies = pd.DataFrame({
    'movie_id': range(n_items),
    'title': [f'Movie_{i}' for i in range(n_items)],
    'genre': np.random.choice(genres, n_items),
    'year': np.random.randint(1990, 2026, n_items),
    'avg_rating': np.random.uniform(2.0, 5.0, n_items).round(1),
})

# User-item interaction matrix (sparse — only 8% filled)
ratings_matrix = np.zeros((n_users, n_items))
for u in range(n_users):
    # Each user rates ~4 movies (8% of 50)
    n_rated = np.random.randint(2, 8)
    rated_items = np.random.choice(n_items, n_rated, replace=False)
    for item in rated_items:
        # Rating depends on genre preference (hidden pattern)
        genre_bonus = 1.0 if movies.iloc[item]['genre'] in ['Action', 'Sci-Fi'] and u < 100 else 0
        genre_bonus += 1.0 if movies.iloc[item]['genre'] in ['Drama', 'Romance'] and u >= 100 else 0
        ratings_matrix[u, item] = np.clip(np.random.normal(3 + genre_bonus, 1), 1, 5).round(1)

# Sparsity analysis
n_ratings = np.count_nonzero(ratings_matrix)
sparsity = 1 - n_ratings / (n_users * n_items)

print(f'Dataset Summary:')
print(f'  Users: {n_users}')
print(f'  Items: {n_items}')
print(f'  Ratings: {n_ratings}')
print(f'  Sparsity: {sparsity:.1%} (typical production: 99%+)')
print(f'\nRatings distribution:')
flat_ratings = ratings_matrix[ratings_matrix > 0]
print(f'  Mean: {flat_ratings.mean():.2f}, Std: {flat_ratings.std():.2f}')
print(f'  Range: [{flat_ratings.min():.1f}, {flat_ratings.max():.1f}]')

---
## 2 · Content-Based Filtering

### Idea
Recommend items SIMILAR to what the user has liked before, based on item features.

```
User liked: "The Matrix" (Sci-Fi, Action)
→ Recommend: "Interstellar" (Sci-Fi), "John Wick" (Action)
```

### How It Works
1. Represent each item as a feature vector (genre, director, keywords)
2. Build a user profile from items they've liked
3. Recommend items most similar to the user profile

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

class ContentBasedRecommender:
    """Content-Based Filtering: recommend items similar to user's history.
    
    Best for: new items with rich metadata (Netflix genres, Spotify audio features).
    Weakness: only recommends 'more of the same' (filter bubble).
    """
    
    def __init__(self, item_features):
        # One-hot encode categorical features
        self.encoder = OneHotEncoder(sparse_output=False)
        self.item_vectors = self.encoder.fit_transform(item_features[['genre']])
        # Add normalized year as a feature
        years = (item_features['year'] - 1990) / 35  # Normalize to [0, 1]
        self.item_vectors = np.hstack([self.item_vectors, years.values.reshape(-1, 1)])
    
    def build_user_profile(self, user_ratings, item_indices):
        """User profile = weighted average of liked item vectors."""
        profile = np.zeros(self.item_vectors.shape[1])
        total_weight = 0
        for item_idx, rating in zip(item_indices, user_ratings):
            if rating > 3.0:  # Only consider positively-rated items
                profile += self.item_vectors[item_idx] * rating
                total_weight += rating
        if total_weight > 0:
            profile /= total_weight
        return profile
    
    def recommend(self, user_profile, already_rated, top_k=5):
        """Find items most similar to user profile."""
        similarities = cosine_similarity(user_profile.reshape(1, -1), self.item_vectors)[0]
        # Exclude already-rated items
        similarities[already_rated] = -1
        top_indices = similarities.argsort()[::-1][:top_k]
        return top_indices, similarities[top_indices]

# Demo
cb = ContentBasedRecommender(movies)

# User 0's profile
user_id = 0
rated_items = np.where(ratings_matrix[user_id] > 0)[0]
user_ratings_list = ratings_matrix[user_id, rated_items]

print(f'User {user_id} rated:')
for item, rating in zip(rated_items, user_ratings_list):
    print(f'  {movies.iloc[item]["title"]:15} ({movies.iloc[item]["genre"]:8}) → {rating}')

# Build profile and recommend
profile = cb.build_user_profile(user_ratings_list, rated_items)
recs, scores = cb.recommend(profile, rated_items, top_k=5)

print(f'\nContent-Based Recommendations:')
for item, score in zip(recs, scores):
    print(f'  {movies.iloc[item]["title"]:15} ({movies.iloc[item]["genre"]:8}) — similarity: {score:.3f}')

print(f'\nNotice: recommendations are in the same genres the user already likes.')
print(f'This is content-based\'s strength AND weakness — no serendipity.')

---
## 3 · Collaborative Filtering

### Idea
Recommend items that SIMILAR USERS have liked.

```
User A liked: Matrix, Inception, Interstellar  
User B liked: Matrix, Inception, Arrival (A doesn't know this movie)
→ Recommend "Arrival" to User A (User B is similar and liked it)
```

### Two Flavors

| Type | Similarity Between | Question Answered |
|------|-------------------|------------------|
| **User-Based CF** | Users | "Who is like me? What did they like?" |
| **Item-Based CF** | Items | "What items are like things I liked?" |

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class CollaborativeFilter:
    """User-Based Collaborative Filtering.
    
    How Amazon's original engine worked (1998).
    Key insight: you don't need to know WHY users are similar,
    only that their rating patterns match.
    """
    
    def __init__(self, ratings_matrix):
        self.R = ratings_matrix
        # Compute user-user similarity (only on co-rated items)
        # Replace 0s (missing) with NaN for proper mean centering
        R_centered = self.R.copy()
        for i in range(R_centered.shape[0]):
            rated = R_centered[i] > 0
            if rated.sum() > 0:
                R_centered[i, rated] -= R_centered[i, rated].mean()
        R_centered[self.R == 0] = 0  # Keep missing as 0
        self.user_sim = cosine_similarity(R_centered)
        np.fill_diagonal(self.user_sim, 0)  # Don't compare with self
    
    def predict(self, user_id, item_id, k=20):
        """Predict rating for (user, item) using k most similar users."""
        # Find k most similar users who rated this item
        item_raters = np.where(self.R[:, item_id] > 0)[0]
        if len(item_raters) == 0:
            return self.R[user_id][self.R[user_id] > 0].mean() if (self.R[user_id] > 0).any() else 3.0
        
        # Sort by similarity to target user
        sims = self.user_sim[user_id, item_raters]
        top_k = min(k, len(item_raters))
        top_indices = sims.argsort()[::-1][:top_k]
        top_users = item_raters[top_indices]
        top_sims = sims[top_indices]
        
        # Weighted average of similar users' ratings
        if np.abs(top_sims).sum() == 0:
            return 3.0
        pred = np.dot(top_sims, self.R[top_users, item_id]) / (np.abs(top_sims).sum() + 1e-8)
        return np.clip(pred, 1, 5)
    
    def recommend(self, user_id, top_k=5):
        """Recommend top-k unrated items."""
        unrated = np.where(self.R[user_id] == 0)[0]
        predictions = [(item, self.predict(user_id, item)) for item in unrated]
        predictions.sort(key=lambda x: x[1], reverse=True)
        return predictions[:top_k]

# Demo
cf = CollaborativeFilter(ratings_matrix)

user_id = 0
print(f'User {user_id} - Similar users (top 5):')
most_similar = cf.user_sim[user_id].argsort()[::-1][:5]
for sim_user in most_similar:
    print(f'  User {sim_user}: similarity = {cf.user_sim[user_id, sim_user]:.3f}')

recommendations = cf.recommend(user_id, top_k=5)
print(f'\nCollaborative Filtering Recommendations:')
for item_id, pred_rating in recommendations:
    print(f'  {movies.iloc[item_id]["title"]:15} ({movies.iloc[item_id]["genre"]:8}) — predicted rating: {pred_rating:.2f}')

print(f'\nCF can recommend ACROSS genres — serendipitous discovery!')
print(f'Weakness: can\'t recommend items with no ratings (cold start).')

---
## 4 · Evaluation Metrics

### Rating Prediction Metrics

| Metric | Formula | When to Use |
|--------|---------|-------------|
| RMSE | √(Σ(predicted - actual)²/n) | Explicit ratings (1-5 stars) |
| MAE | Σ|predicted - actual|/n | More interpretable |

### Ranking Metrics (More Important in Production!)

| Metric | What It Measures | Why It Matters |
|--------|-----------------|----------------|
| **Precision@K** | % of top-K recommendations that are relevant | Are we recommending good items? |
| **Recall@K** | % of relevant items that appear in top-K | Are we finding all good items? |
| **NDCG@K** | Quality of ranking (higher = better ordering) | Are the BEST items ranked first? |
| **MAP** | Average precision across all users | Overall ranking quality |
| **Hit Rate@K** | % of users with at least 1 relevant rec in top-K | Basic satisfaction measure |

In [ ]:
import numpy as np

def evaluate_recommender(model, ratings_matrix, n_test_users=50):
    """Evaluate with leave-one-out: hide one rating per user, try to recover it."""
    hits = 0
    rmse_errors = []
    ndcg_scores = []
    
    for user_id in range(min(n_test_users, ratings_matrix.shape[0])):
        rated_items = np.where(ratings_matrix[user_id] > 0)[0]
        if len(rated_items) < 2:
            continue
        
        # Hide the last rated item
        test_item = rated_items[-1]
        actual_rating = ratings_matrix[user_id, test_item]
        
        # Predict
        predicted_rating = model.predict(user_id, test_item)
        rmse_errors.append((predicted_rating - actual_rating) ** 2)
        
        # Hit@10: is the test item in top-10 recommendations?
        recs = model.recommend(user_id, top_k=10)
        rec_items = [r[0] for r in recs]
        if test_item in rec_items:
            hits += 1
    
    rmse = np.sqrt(np.mean(rmse_errors)) if rmse_errors else 0
    hit_rate = hits / n_test_users
    
    return {'RMSE': rmse, 'Hit@10': hit_rate}

results = evaluate_recommender(cf, ratings_matrix)
print(f'Collaborative Filtering Evaluation:')
print(f'  RMSE:    {results["RMSE"]:.3f} (lower is better)')
print(f'  Hit@10:  {results["Hit@10"]:.1%} (% of test items found in top-10)')
print(f'\nIn production, Hit@10 > 10% is decent for sparse data.')
print(f'Netflix Prize-winning approaches achieved RMSE ~0.85 on a 1-5 scale.')

---
## 🎓 DEEP QUESTION ANSWERED

**Q:** *How does a RecSys fill in 99.7% missing data?*

**A:** It doesn't fill ALL of them — it strategically estimates the ones that matter (likely high ratings for each user). Collaborative filtering leverages the pattern that users with similar taste tend to agree. If 50 users similar to you all rated "Inception" 5 stars, it's a safe bet you'll like it too. The system only needs a FEW ratings per user to find these taste clusters — the correlation structure in the data does the rest.

---

## ✅ Knowledge Check

### Q1: Content-based vs collaborative filtering
A brand-new movie with no ratings just launched. Which approach can recommend it? Why?

<details><summary>👉 Answer</summary>

Content-based CAN recommend it — it only needs the movie's features (genre, director, actors). Collaborative filtering CANNOT — it needs other users' ratings to find patterns. This is the **cold start problem**, and it's why production systems need hybrid approaches.
</details>

### Q2: User-based vs item-based CF
Amazon uses item-based CF instead of user-based CF. Why is it better for their scale?

<details><summary>👉 Answer</summary>

Item-item similarities are MORE STABLE than user-user similarities (users' tastes change, items don't). Also, item-item similarity can be pre-computed offline (millions of items × items), while user-user similarity changes with every new rating. At Amazon's scale (300M+ users), recomputing user-user similarity is prohibitively expensive.
</details>

### Q3: Evaluation trap
Your model has RMSE=0.80 (great!). But users say the recommendations are boring. What went wrong?

<details><summary>👉 Answer</summary>

RMSE measures rating accuracy, not user satisfaction. The model might be recommending safe, popular items (high predicted rating) but not INTERESTING ones. Production RecSys also optimizes for diversity, novelty, and serendipity — not just accuracy. This is why ranking metrics (NDCG, Hit@K) and online A/B tests matter more than offline RMSE.
</details>

---

## 🔨 Practical Practice

### Exercise 1: Item-Based CF
Implement item-based collaborative filtering: compute item-item similarity matrix, and for each user, recommend items similar to their highly-rated items. Compare to user-based CF.

### Exercise 2: Hybrid Recommender
Combine content-based and collaborative filtering scores: `hybrid_score = α * content_score + (1-α) * cf_score`. Test different α values from 0 to 1.

### Exercise 3: Cold Start Solution
A new user signs up with zero ratings. Design a strategy to bootstrap their profile using: (1) popular items, (2) diversity sampling, (3) interactive onboarding.

---

**Next →** RS 02: Matrix Factorization & Embeddings